# 05.15 - Cross-Validation & Hyperparameter Tuning

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

How do we reliably estimate model performance and find the best hyperparameters? We use **k-fold cross-validation** and **grid/random search**.

## 2. Why Does This Matter?

A single train/test split can be misleading. Cross-validation gives a more reliable estimate. Hyperparameter tuning finds the best model configuration.

## 3. Prerequisites

- Unit 05.4 (Model Evaluation), Unit 05.6 (Random Forests)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain k-fold cross-validation
- Use grid search
- Use random search
- Avoid overfitting during tuning

## 5. Mental Model

k-fold CV:

1. Split data into k folds.
2. For each fold, train on k-1 folds, validate on 1.
3. Average the k validation scores.

Grid search: try all combinations of hyperparameters.
Random search: try random combinations.

Always tune on training data (via CV), never on test.


## 6. Generate Data

Use a classification dataset.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")


Train: 700, Test: 300


## 7. Single Split vs Cross-Validation

A single split can be noisy. CV averages over multiple splits.


In [2]:
# Single split
rf = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
single_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Single split accuracy: {single_acc:.3f}")

# 5-fold CV
cv_scores = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=42), X_train, y_train, cv=5)
print(f"5-fold CV accuracy: {cv_scores}")
print(f"Mean CV: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
print("\nCV gives a more reliable estimate with uncertainty.")


Single split accuracy: 0.910


5-fold CV accuracy: [0.93571429 0.92142857 0.92857143 0.87142857 0.92142857]
Mean CV: 0.916 +/- 0.023

CV gives a more reliable estimate with uncertainty.


## 8. Grid Search

Search over a grid of hyperparameters.


In [3]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_:.3f}")
print(f"Test accuracy: {accuracy_score(y_test, grid.predict(X_test)):.3f}")


Best params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV score: 0.930
Test accuracy: 0.917


## 9. Random Search

Random search samples random combinations - often more efficient.


In [4]:
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 10),
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist, n_iter=20, cv=3, scoring='accuracy', random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)
print(f"Best params: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.3f}")
print(f"Test accuracy: {accuracy_score(y_test, random_search.predict(X_test)):.3f}")


Best params: {'max_depth': 14, 'min_samples_split': 2, 'n_estimators': 98}
Best CV score: 0.930
Test accuracy: 0.910


## 10. Effect of Number of Folds

More folds = more data per training, but more computation.


In [5]:
for cv in [2, 5, 10]:
    scores = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=42), X_train, y_train, cv=cv)
    print(f"cv={cv:2d}: mean={scores.mean():.3f}, std={scores.std():.3f}")
print("\nMore folds give a more stable estimate but cost more.")


cv= 2: mean=0.901, std=0.013


cv= 5: mean=0.916, std=0.023


cv=10: mean=0.924, std=0.036

More folds give a more stable estimate but cost more.


## 11. Failure Case: Tuning on Test Set

If you tune on the test set, you overfit to it.


In [6]:
print("If you repeatedly evaluate on the test set during tuning:")
print("  - You overfit to the test set.")
print("  - The test accuracy becomes optimistic.")
print("  - The model may fail on new data.")
print("\nAlways tune on training data (via CV) and test once at the end.")


If you repeatedly evaluate on the test set during tuning:
  - You overfit to the test set.
  - The test accuracy becomes optimistic.
  - The model may fail on new data.

Always tune on training data (via CV) and test once at the end.


## 12. Debugging: Common Errors

- **Tuning on test**: overfitting to test.
- **Too few folds**: noisy estimate.
- **Leakage**: preprocessing before CV split.

## 13. Real-World Considerations

- Use CV for reliable estimates.
- Random search is often more efficient than grid.
- Use a separate validation set for early stopping.

## 14. Common Mistakes

- Tuning on test.
- Using a single split.
- Not stratifying for imbalanced data.

## 15. When NOT to Use

- When data is tiny (CV folds too small).
- When computation is prohibitive.

## 16. Challenge

Use GridSearchCV with a pipeline that includes scaling, and report the best parameters.


In [7]:
# Challenge: tune a pipeline
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pipe = make_pipeline(StandardScaler(), SVC())
param_grid = {
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['linear', 'rbf'],
}
grid_pipe = GridSearchCV(pipe, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_pipe.fit(X_train, y_train)
print(f"Best params: {grid_pipe.best_params_}")
print(f"Best CV score: {grid_pipe.best_score_:.3f}")
print(f"Test accuracy: {accuracy_score(y_test, grid_pipe.predict(X_test)):.3f}")
print("\nTuning a pipeline ensures preprocessing is part of CV.")


Best params: {'svc__C': 1, 'svc__kernel': 'rbf'}
Best CV score: 0.933
Test accuracy: 0.950

Tuning a pipeline ensures preprocessing is part of CV.


## 17. Closed-Book Recall

Without looking back:

1. How does k-fold CV work?
2. What is the difference between grid and random search?
3. Why tune on training data, not test?
4. What is the risk of tuning on test?

## 18. Teach-Back Questions

Explain to another person:

- Why cross-validation is more reliable than a single split.
- The difference between grid and random search.

## 19. Summary

You used k-fold CV, grid search, and random search to tune models reliably, and learned to avoid overfitting during tuning.

## 20. Further Experiment

- Try Bayesian optimization.
- Use nested CV for unbiased evaluation.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn, scipy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
